# Week 1 Exercise 1.2

In [1]:
import numpy as np
import math

from perceptron import Perceptron
from activation import ActivationFunction

1. Fill in the Sigmoid and LinearActivation classes in mlp.py.

In [38]:
class Sigmoid(ActivationFunction):
    """ 
        Sigmoid activation: `f(x) = 1/(1+e^(-x))`
    """
    def forward(self, x):
        """
            Activation function output.
            TODO: Change the function to return the correct value, given input `x`.
        """
        f = 1/(1 + np.exp(-x))
        return f

    def gradient(self, x):
        """
            Activation function derivative.
            TODO: Change the function to return the correct value, given input `x`.
        """
        return x * (1 - x)

class LinearActivation(ActivationFunction):
    """ 
        Linear activation: `f(x) = x`
    """
    def forward(self, x):
        """
            Activation function output.
            TODO: Change the function to return the correct value, given input `x`.
        """
        f = int(x>0)
        return f

    def gradient(self, x):
        """
            Activation function derivative.
            TODO: Change the function to return the correct value, given input `x`.
        """
        return 1

In [3]:
actF = LinearActivation()
print(actF.forward(1))  # Returns 1
print(actF.forward(0))  # Returns 0
print(actF.forward(-1))  # Returns 0

actF = Sigmoid()
print(actF.forward(1))  # Returns 1
print(actF.forward(0))  # Returns 0
print(actF.forward(-1))  # Returns 0

1
0
0
0.7310585786300049
0.5
0.2689414213699951


Linear function was the same as the SignActivation class from before and then the sigmoid function was used for the Sigmoid class. Both of them work since the Linear Function returns either 0 or 1 and the Sigmoid function returns a value between 0 and 1.

2. Now we will create a Layer class which contains a number of perceptrons.

In [4]:
class Layer:
    def __init__(self, num_inputs, num_units, act_f):
        """ 
            Initialize the layer, creating `num_units` perceptrons with `num_inputs` each. 
        """
        self.ps = []
        for i in range(num_units):
            self.ps.append(Perceptron(num_inputs, act_f, 0.5))
        
        # self.ps.append(Perceptron(num_inputs, LinearActivation, 0.5))

    def activation(self, x):
        """ Returns the activation `a` of all perceptrons in the layer, given the input vector`x`. """
        return np.array([p.activation(x) for p in self.ps])

    def output(self, a):
        """ Returns the output `o` of all perceptrons in the layer, given the activation vector `a`. """
        return np.array([p.output(ai) for p, ai in zip(self.ps, a)])

    def predict(self, x):
        """ Returns the output `o` of all perceptrons in the layer, given the input vector `x`. """
        return np.array([p.predict(x) for p in self.ps])

    def gradient(self, a):
        """ Returns the gradient of the activation function for all perceptrons in the layer, given the activation vector `a`. """
        return np.array([p.gradient(ai) for p, ai in zip(self.ps, a)])

    def update_weights(self, dw):
        """ 
        Update the weights of all of the perceptrons in the layer, given the weight change of each.
        Input size: (n_inputs+1, n_units)
        """
        for i in range(self.num_units):
            self.ps[i].w += dw[:,i]
    
    @property
    def w(self):
        """
            Returns the weights of the neurons in the layer.
            Size: (n_inputs+1, n_units)
        """
        return np.array([p.w for p in self.ps]).T

    def import_weights(self, w):
        """ 
            Import the weights of all of the perceptrons in the layer.
            Input size: (n_inputs+1, n_units)
        """
        for i in range(self.num_units):
            self.ps[i].w = w[:,i]

In [5]:
print("Testing Layer with transpose")
data = np.array([math.pi, 1]).T
lay = Layer(len(data), 5, Sigmoid)
lay.predict(data)
print(lay.w)

# Without transpose
print("Testing Layer without transpose")
data = np.array([math.pi, 1])
lay = Layer(len(data), 5, Sigmoid)
lay.predict(data)
print(lay.w)

Testing Layer with transpose
[[ 0.30786742 -0.57808569 -0.39376749  0.02365762 -0.15404051]
 [-0.09438642  0.75998159 -0.28038999  0.0696598  -0.59464016]]
Testing Layer without transpose
[[ 0.12629118 -0.04639255  0.25862441  0.29938995 -0.00683711]
 [ 0.13735804 -0.40958857 -0.48930374 -0.38452656 -0.22721563]]


Here the init was written for the Layer. A for loop goes through the number of units wanted (5 in this test case) and creates a Perceptron using the desired activation function from var act_f with a certain number of inputs each.

3. Now we will deﬁne the MLP.

In [6]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(self.num_inputs, self.n_hidden_units, Sigmoid) # hidden layer 1
        self.l_out = Layer(self.n_hidden_units, self.n_outputs, LinearActivation) # output layer

It has 2 inputs into the output layer since it had two inputs to begin with and our layers neveer change the number of inputs before entering the output (linear) layer which then turns into one output of 0 or 1.

4. Implement the predict function using the functions deﬁned for each layer.

In [7]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(self.num_inputs, self.n_hidden_units, Sigmoid) # hidden layer 1
        self.l_out = Layer(self.n_hidden_units, self.n_outputs, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        ## Layer 1
        a = self.l1.predict(x)
        # a1 = self.l1.activation(input)
        #print(f"Activations for layer 1: {a1}")
        # o1 = self.l1.output(a1)
        #print(f"Outputs for layer 1: {o1}")

        ## Layer 2
        output = self.l_out.predict(a)
        # a2 = self.l2.activation(o1)
        #print(f"Activations for layer 2: {a2}")
        # y = self.l2.output(a2)
        #print(f"Output of output layer: {y}")
        
        return output

In [8]:
num_inputs = 2
n_hidden_units = 2
n_outputs = 1
mlp = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)
mlp.predict(data)

array([1])

This calls the predict for each of the two layers we have. The commented out code goes step by step through the activation and output to see the values at each step but the easiest implementation is just to call the predict function for each layer where those same activation and output functions are called (in case the predict function works differently for the layer being used).

5. Implement the function calc prediction error.

In [17]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(self.num_inputs, self.n_hidden_units, Sigmoid) # hidden layer 1
        self.l_out = Layer(self.n_hidden_units, self.n_outputs, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        a = self.l1.predict(x)
        output = self.l_out.predict(a)
        return output

    def calc_prediction_error(self, model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            output = model.predict(input)
            print(f"OUTPUT: {output}, TARGET: {target}")
            delta_w = (target - output)
            print(f"ERROR: {delta_w}")
            E = E + delta_w**2
        E = E/len(t)
        return E

In [18]:
data = np.array( [ [0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]
mlp2 = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)
print(mlp2.calc_prediction_error(mlp, xdata, ydata))

OUTPUT: [1], TARGET: 0.0
ERROR: [-1.]
OUTPUT: [1], TARGET: 0.0
ERROR: [-1.]
OUTPUT: [1], TARGET: 0.0
ERROR: [-1.]
OUTPUT: [1], TARGET: 1.0
ERROR: [0.]
OUTPUT: [1], TARGET: 1.0
ERROR: [0.]
OUTPUT: [1], TARGET: 1.0
ERROR: [0.]
[0.5]


Mean Square Error function has been implemented using equation 6 in the worksheet. The tests show that the equation is implemented correctly and also underlines why we took the absolute value of the prediction in part 1.1.4. The overall error is 0.5 since half of the predictions produce an error and half don't.

6. Implement the function MLP.train.

a. Calculate the prediction (forward pass)

In [31]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(num_inputs, 1, Sigmoid) # hidden layer 1
        self.l_out = Layer(num_inputs, 1, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        # print(f"mlp: {x}")
        a = self.l1.predict(x)
        print(a)
        # print("done with hidden")
        output = self.l_out.predict(a)
        return output

    def train(self, inputs, outputs):
        """
            Train the network

        Parameters
        ----------
        `x` : numpy array
            Inputs (size: n_examples, n_inputs)
        `t` : numpy array
            Targets (size: n_examples, n_outputs)

        TODO: Write the function to iterate through training examples and apply gradient descent to update the neuron weights
        """

        delta1 = np.array([[0.0,0.0],[0.0,0.0],[0.0,0.0]])
        delta_out = np.array([[0.0],[0.0],[0.0]])

        a_array = []
        o_array = []

        # Loop over training examples
        for input, target in zip(inputs, outputs):
            # Forward pass
            a1 = self.l1.activation(input)
            # print(f"Activations for layer 1: {a1}")
            a_array.append(a1[0])
            o1 = self.l1.output(a1)
            # print(f"Outputs for layer 1: {o1}")
            o_array.append(o1[0])

            a2 = self.l_out.activation(o1)
            # print(f"Activations for layer 2: {a2}")
            a_array.append(a2[0])
            y = self.l_out.output(a2)
            o_array.append(y[0])
            # print(f"Output of output layer: {y}")

        print(f"Here are all the activation values: {a_array}")
        print(f"Here are all the output values: {o_array}")

    def calc_prediction_error(self, model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            output = model.predict(input)
            print(f"OUTPUT: {output}, TARGET: {target}")
            delta_w = (target - output)
            print(f"ERROR: {delta_w}")
            E = E + delta_w**2
        E = E/len(t)
        return E

In [32]:
data = np.array( [ [0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]
mlp3 = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)

num_epochs = 5
for epoch in range(num_epochs):
    print(f"EPOCH: {epoch}")
    mlp3.train(xdata, ydata)

EPOCH: 0
Here are all the activation values: [0.11308605641955902, 0.2641207117503286, 0.5, 0.3112296656009273, 0.17851633851735405, 0.27225547041757453, -0.27382788716088197, 0.21598379837508472, -0.5476557743217639, 0.18320422904393646, -0.1024213517539404, 0.23720851111151853]
Here are all the output values: [0.5282414235006572, 1, 0.6224593312018546, 1, 0.5445109408351491, 1, 0.43196759675016944, 1, 0.3664084580878729, 1, 0.47441702222303705, 1]
EPOCH: 1
Here are all the activation values: [0.11308605641955902, 0.2641207117503286, 0.5, 0.3112296656009273, 0.17851633851735405, 0.27225547041757453, -0.27382788716088197, 0.21598379837508472, -0.5476557743217639, 0.18320422904393646, -0.1024213517539404, 0.23720851111151853]
Here are all the output values: [0.5282414235006572, 1, 0.6224593312018546, 1, 0.5445109408351491, 1, 0.43196759675016944, 1, 0.3664084580878729, 1, 0.47441702222303705, 1]
EPOCH: 2
Here are all the activation values: [0.11308605641955902, 0.2641207117503286, 0.5, 

Here is the training using only the activation and output functions for each layer. No prediction function is used so that the proper data can be collected at each step and used later on.

b. From the prediction, calculate the error for the output layer, using eq. (7) and the errors for the previous layer using eq. (8)

In [35]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(num_inputs, 1, Sigmoid) # hidden layer 1
        self.l_out = Layer(num_inputs, 1, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        # print(f"mlp: {x}")
        a = self.l1.predict(x)
        print(a)
        # print("done with hidden")
        output = self.l_out.predict(a)
        return output

    def export_weights(self):
        return [self.l1.w, self.l_out.w]

    def import_weights(self, ws):
        if ws[0].shape == (self.n_inputs+1, self.n_hidden_units) and ws[1].shape == (self.n_hidden_units+1, 1):
            #print("Importing weights..")
            self.l1.import_weights(ws[0])
            self.l_out.import_weights(ws[1])        
        else:
            print("Sizes do not match")

    def train(self, inputs, outputs):
        """
            Train the network

        Parameters
        ----------
        `x` : numpy array
            Inputs (size: n_examples, n_inputs)
        `t` : numpy array
            Targets (size: n_examples, n_outputs)

        TODO: Write the function to iterate through training examples and apply gradient descent to update the neuron weights
        """

        # delta1 = np.array([[0.0,0.0],[0.0,0.0],[0.0,0.0]])
        # delta_out = np.array([[0.0],[0.0],[0.0]])
        w_hidden = np.random.uniform(size=(self.num_inputs, self.n_hidden_units))
        w_output = np.random.uniform(size=(self.n_hidden_units, self.n_outputs))

        a_array = []
        o_array = []

        r = self.alpha      # Learning rate

        # weights = self.import_weights()

        # Loop over training examples
        for input, target in zip(inputs, outputs):
            # Forward pass
            a1 = self.l1.activation(input)
            # print(f"Activations for layer 1: {a1}")
            a_array.append(a1[0])
            o1 = self.l1.output(a1)
            # print(f"Outputs for layer 1: {o1}")
            o_array.append(o1[0])
            # delta1_error = o1-target

            a2 = self.l_out.activation(o1)
            # print(f"Activations for layer 2: {a2}")
            a_array.append(a2[0])
            y = self.l_out.output(a2)
            # print(f"Output of output layer: {y}")
            o_array.append(y[0])

            ## Error calculation
            error1 = target - o1
            error2 = target-y

            ## Backpropogation
            # Layer 1
            dJ1 = self.l1.gradient(a1)*error1
            # print(f"ERROR: {dJ1}")
            w_output += o1.T.dot(dJ1)
            # print(f"WEIGHTS: {w_output}")

            # Layer 2
            dJ = self.l_out.gradient(a2)*error2
            # print(f"ERROR: {dJ}")
            w_output += y.T.dot(dJ)
            # print(f"ERROR: {w_output}")

        # print(f"Here are all the activation values: {a_array}")
        # print(f"Here are all the output values: {o_array}")
            
    def calc_prediction_error(self, model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            output = model.predict(input)
            print(f"OUTPUT: {output}, TARGET: {target}")
            delta_w = (target - output)
            print(f"ERROR: {delta_w}")
            E = E + delta_w**2
        E = E/len(t)
        return E

In [36]:
data = np.array( [ [0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]
mlp3 = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)

num_epochs = 5
for epoch in range(num_epochs):
    print(f"EPOCH: {epoch}")
    mlp3.train(xdata, ydata)

EPOCH: 0
ERROR: [-0.05103663]
WEIGHTS: [[0.70621925]
 [0.40992327]]
ERROR: [-0.1]
WEIGHTS: [[0.60621925]
 [0.30992327]]
ERROR: [-0.06224593]
WEIGHTS: [[0.56747369]
 [0.27117771]]
ERROR: [-0.1]
WEIGHTS: [[0.46747369]
 [0.17117771]]
ERROR: [-0.04375346]
WEIGHTS: [[0.44833004]
 [0.15203406]]
ERROR: [-0.1]
WEIGHTS: [[0.34833004]
 [0.05203406]]
ERROR: [0.0602779]
WEIGHTS: [[0.37227369]
 [0.07597771]]
ERROR: [0.]
WEIGHTS: [[0.37227369]
 [0.07597771]]
ERROR: [0.06972245]
WEIGHTS: [[0.39338393]
 [0.09708795]]
ERROR: [0.]
WEIGHTS: [[0.39338393]
 [0.09708795]]
ERROR: [0.06028915]
WEIGHTS: [[0.41732527]
 [0.12102929]]
ERROR: [0.]
WEIGHTS: [[0.41732527]
 [0.12102929]]
EPOCH: 1
ERROR: [-0.05103663]
WEIGHTS: [[0.25071264]
 [0.34546377]]
ERROR: [-0.1]
WEIGHTS: [[0.15071264]
 [0.24546377]]
ERROR: [-0.06224593]
WEIGHTS: [[0.11196708]
 [0.20671821]]
ERROR: [-0.1]
WEIGHTS: [[0.01196708]
 [0.10671821]]
ERROR: [-0.04375346]
WEIGHTS: [[-0.00717657]
 [ 0.08757456]]
ERROR: [-0.1]
WEIGHTS: [[-0.10717657]
 [-0.

Here we have implemented equations 7 and 8 to calculate the error in backpropogation.

c. Calculate the weight change in each layer from eq. (9).

Here is our first iteration of the function

In [72]:
class MLP:
     """ 
         Multi-layer perceptron class
     Parameters
     ----------
     n_inputs : int
         Number of inputs
     n_hidden_units : int
         Number of units in the hidden layer
     n_outputs : int
         Number of outputs (One output -> one unit in layer)
     alpha : float
         Learning rate used for gradient descent
     """
     def __init__(self, n_inputs, n_hidden_units, n_outputs, alpha):
        self.n_inputs = n_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs
        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(self.n_inputs, self.n_hidden_units, Sigmoid) # Hidden layer 1
        self.l2 = Layer(self.n_hidden_units, self.n_outputs, LinearActivation) # Output layer has one unit with as many inputs as there are units in hidden layer

     def predict(self, x):
        l1_out = self.l1.predict(x)
        #print(f"L1 prediction: {l1_out}")
        output = self.l2.predict(l1_out)
        #print(f"L2 prediction: {output}")

        return output

     def train(self, x, t, N):
        """
            Train the network

        Parameters
        ----------
         `x` : numpy array
             Inputs (size: n_examples, n_inputs)
         `t` : numpy array
             Targets (size: n_examples, n_outputs)

        TODO: Write the function to iterate through training examples and apply gradient descent to update the neuron weights
         """
        #percepLin = Perceptron(num_inputs, Linear, 0.5)
        #percepSig = Perceptron(num_inputs, Sigmoid, 0.5)

        p = 0
        dw_l1 = np.array([[0.0,0.0],[0.0,0.0],[0.0,0.0]]) # temporary array for weight changes
        dw_l2 = np.array([[0.0],[0.0],[0.0]]) # temporary array for weight changes

        # Loop over training examples
        for input, target in zip(x, t):     # x = training data, t = training target
            # Forward pass
            #print(f"Training set: {p} ------------------------------------------")
            p = p + 1
            #print("Forwards pass:")
            a1 = self.l1.activation(input)
            #print(f"Activations for layer 1: {a1}")
            o1 = self.l1.output(a1)
            #print(f"Outputs for layer 1: {o1}")

            a2 = self.l2.activation(o1)
            #print(f"Activations for layer 2: {a2}")
            y = self.l2.output(a2)
            #print(f"Output of output layer: {y}")
            

            # Backpropagation
            # Error of output layer:
            sig_K_j = self.l2.gradient(a2)*(y-target) # = g'K(aK)(y-t)
            #print(f"sig_K_j: {sig_K_j}")
            
            # Errors of first (previous) layer (simplified eq. 8 because there is only one unit in next layer (output))
            sig_k_j = self.l1.gradient(a1) * self.l2.w[1] * sig_K_j
            #print(f"sig_k_1: {sig_k_j[0]}")
            #print(f"sig_k_2: {sig_k_j[1]}")

            # Calculate and sum weight changes for each input to each neuron in layer 1 and 2:
            # Layer 1 (Hidden)
            o = np.append(1,input) # add bias input x0 to currrent training set input
            for i in range(self.n_inputs+1):
                for j in range(self.n_hidden_units):
                    # for every neuron, a weight change is calculated for each input
                    dw_l1[i,j] += -self.alpha/N * sig_k_j[j] * o[i] # Naming convention: L1 = layer 1, [0,1] = input 0 (bias) of neutron 1

            # Layer 2 (Output)
            o_out = np.append(1,o1) # add bias input x0 to currrent training set input
            for i in range(self.n_inputs+1):
                # for every neuron, a weight change is calculated for each input
                dw_l2[i] += -self.alpha/N * sig_K_j * o_out[i]

        # Apply weight updates
        print("layer 1 weights before contribution:")
        print(self.l1.w)
        print("layer 2 weights before contribution:")
        print(self.l2.w)

        ws1 = self.l1.w.T + dw_l1
        ws2 = self.l2.w.T + dw_l2

        #print("layer 1 weights after contribution:")
        #print(ws1)
        #print("layer 2 weights after contribution:")
        #print(ws2)

        self.import_weights([ws1, ws2])
        return None # remove this line

     def export_weights(self):
        return [self.l1.w, self.l2.w]


     def import_weights(self, ws):
        if ws[0].shape == (self.n_inputs+1, self.n_hidden_units) and ws[1].shape == (self.n_hidden_units+1, 1):
            #print("Importing weights..")
            self.l1.import_weights(ws[0])
            self.l2.import_weights(ws[1])        
        else:
            print("Sizes do not match")


     def calc_prediction_error(model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            #print(f"MSE input: {input}")
            output = model.predict(input)
            #print(f"OUTPUT: {output}")
            delta_w = (target - output)
            #print(f"ERROR: {delta_w}")
            E = E + delta_w**2/len(t)
        return E

In [73]:
data = np.array( [[0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]

num_inputs = 2
n_hidden_units = 2
n_outputs = 1
mlp = MLP(num_inputs, n_hidden_units, n_outputs, 1e-3)

acceptable = 0
j = 0

while(not(acceptable) and j < 5000):
    print("Epoch:", j)
    print("Test MLP training:")
    mlp.train(xdata, ydata, 6) # Train new weights
    mse = mlp.calc_prediction_error(xdata, ydata) # Make a prediction with new weights and find error
    print(f"MSE: {mse}")

    acceptable = np.all(abs(mse)<0.23) # continue reiterating if prediction is not acceptable


    j = j + 1
    print(j)

print(f"Layer 1: {mlp.l1.w}")
print(f"Layer 2: {mlp.l2.w}")
print(f"Training took {j} iterations")

Epoch: 0
Test MLP training:
layer 1 weights before contribution:
[[ 0.20778546 -0.58454048]
 [ 0.81653183 -0.08854742]]
layer 2 weights before contribution:
[[-0.49071633]
 [ 0.27515951]]


ValueError: operands could not be broadcast together with shapes (2,2) (3,2) 

Since this was taking a long time to run (and for some reason is not working in the notebook) we decided to try and reimplement the function

In [99]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(num_inputs, 1, Sigmoid) # hidden layer 1
        self.l_out = Layer(num_inputs, 1, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        # print(f"mlp: {x}")
        a = self.l1.predict(x)
        print(a)
        # print("done with hidden")
        output = self.l_out.predict(a)
        return output

    def export_weights(self):
        return [self.l1.w, self.l_out.w]

    def import_weights(self, ws):
        if ws[0].shape == (self.n_inputs+1, self.n_hidden_units) and ws[1].shape == (self.n_hidden_units+1, 1):
            #print("Importing weights..")
            self.l1.import_weights(ws[0])
            self.l_out.import_weights(ws[1])        
        else:
            print("Sizes do not match")

    def train(self, inputs, outputs):
        """
            Train the network

        Parameters
        ----------
        `x` : numpy array
            Inputs (size: n_examples, n_inputs)
        `t` : numpy array
            Targets (size: n_examples, n_outputs)

        TODO: Write the function to iterate through training examples and apply gradient descent to update the neuron weights
        """

        # delta1 = np.array([[0.0,0.0],[0.0,0.0],[0.0,0.0]])
        # delta_out = np.array([[0.0],[0.0],[0.0]])
        w_hidden = np.random.uniform(size=(self.num_inputs, self.n_hidden_units))
        w_output = np.random.uniform(size=(self.n_hidden_units, self.n_outputs))

        d_hidden = np.random.uniform(size=(self.num_inputs, self.n_hidden_units))
        d_output = np.random.uniform(size=(self.n_hidden_units, self.n_outputs))

        a_array = []
        o_array = []

        r = self.alpha      # Learning rate
        N = len(inputs)

        w1 = self.l1.w
        w2 = self.l_out.w

        # weights = self.import_weights()

        # Loop over training examples
        for input, target in zip(inputs, outputs):
            # Forward pass
            a1 = self.l1.activation(input)
            # print(f"Activations for layer 1: {a1}")
            a_array.append(a1[0])
            o1 = self.l1.output(a1)
            # print(f"Outputs for layer 1: {o1}")
            o_array.append(o1[0])
            # delta1_error = o1-target

            a2 = self.l_out.activation(o1)
            # print(f"Activations for layer 2: {a2}")
            a_array.append(a2[0])
            y = self.l_out.output(a2)
            # print(f"Output of output layer: {y}")
            o_array.append(y[0])

            ## Error calculation
            error1 = target - o1
            error2 = target-y

            ## Backpropogation
            # Layer 1
            dJ1 = self.l1.gradient(a1)*error1
            d_hidden += dJ1
            w_hidden += o1
            # w_hidden += (-r/N)*o1.T.dot(dJ1)

            # Layer 2
            dJ2 = self.l_out.gradient(a2)*error2
            # print(f"ERROR: {dJ2}")
            d_output += dJ2
            w_output += y
            # w_output += (-r/N)*y.T.dot(dJ2)

        # print(f"hidden: w {w_hidden}, d {d_hidden}")
        # print(f"output: w {w_output}, d {d_output}")

        # hidden_sum = w_hidden.T.dot(d_hidden)
        # # print(f"hidden_sum: {hidden_sum}")
        # w1 = w1 + np.sum((-r/N)*d_hidden.T.dot(hidden_sum))
        # # output_sum = w_output.T.dot(d_output)
        # # print(f"output_sum: {output_sum}")
        # # w2 = (-r/N)*d_output.T.dot(output_sum)
        # # output_sum = w_output.T.dot(d_output)
        # # print(f"w_output: {w_output}")
        # w2 = w2 + np.sum((-r/N)*d_output.T.dot(w_output))
        # # self.l1.w += w_hidden
        # # self.l_out += w_output
        # print(f"RESULTS: {w1}, {w2}")
            
        

        hidden_sum = w_hidden.T.dot(d_hidden)
        w1 = w1 + np.sum((-r/N)*d_hidden.T.dot(hidden_sum))
        w2 = w2 + np.sum((-r/N)*d_output.T.dot(w_output)) + w1
        # print(f"{w1}, {w2}")
        print(f"RESULTS: {w2}")

        # print(f"Here are all the activation values: {a_array}")
        # print(f"Here are all the output values: {o_array}")
            
    def calc_prediction_error(self, model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            output = model.predict(input)
            print(f"OUTPUT: {output}, TARGET: {target}")
            delta_w = (target - output)
            print(f"ERROR: {delta_w}")
            E = E + delta_w**2
        E = E/len(t)
        return E

In [102]:
data = np.array( [ [0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]
mlp3 = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)

num_epochs = 50
for epoch in range(num_epochs):
    print(f"EPOCH: {epoch}")
    mlp3.train(xdata, ydata)
    print(f"MSE: {mlp.calc_prediction_error(xdata, ydata)}")

EPOCH: 0
RESULTS: [[-1.39485232]
 [ 0.28513667]]
MSE: [0.5]
EPOCH: 1
RESULTS: [[-1.39765478]
 [ 0.28233421]]
MSE: [0.5]
EPOCH: 2
RESULTS: [[-1.39776933]
 [ 0.28221966]]
MSE: [0.5]
EPOCH: 3
RESULTS: [[-1.39946176]
 [ 0.28052723]]
MSE: [0.5]
EPOCH: 4
RESULTS: [[-1.39778705]
 [ 0.28220194]]
MSE: [0.5]
EPOCH: 5
RESULTS: [[-1.40019527]
 [ 0.27979373]]
MSE: [0.5]
EPOCH: 6
RESULTS: [[-1.39775562]
 [ 0.28223337]]
MSE: [0.5]
EPOCH: 7
RESULTS: [[-1.39488329]
 [ 0.2851057 ]]
MSE: [0.5]
EPOCH: 8
RESULTS: [[-1.40519932]
 [ 0.27478967]]
MSE: [0.5]
EPOCH: 9
RESULTS: [[-1.40508681]
 [ 0.27490218]]
MSE: [0.5]
EPOCH: 10
RESULTS: [[-1.40218278]
 [ 0.27780622]]
MSE: [0.5]
EPOCH: 11
RESULTS: [[-1.39396919]
 [ 0.2860198 ]]
MSE: [0.5]
EPOCH: 12
RESULTS: [[-1.40514169]
 [ 0.2748473 ]]
MSE: [0.5]
EPOCH: 13
RESULTS: [[-1.41098397]
 [ 0.26900502]]
MSE: [0.5]
EPOCH: 14
RESULTS: [[-1.38629758]
 [ 0.29369141]]
MSE: [0.5]
EPOCH: 15
RESULTS: [[-1.4105716]
 [ 0.2694174]]
MSE: [0.5]
EPOCH: 16
RESULTS: [[-1.39442623]
 [

Yet another version here below since the results were not what we expected (MSE doesn't change at all)

In [95]:
class MLP:
    """ 
        Multi-layer perceptron class

    Parameters
    ----------
    n_inputs : int
        Number of inputs
    n_hidden_units : int
        Number of units in the hidden layer
    n_outputs : int
        Number of outputs
    alpha : float
        Learning rate used for gradient descent
    """
    def __init__(self, num_inputs, n_hidden_units, n_outputs, alpha=1e-3):
        self.num_inputs = num_inputs
        self.n_hidden_units = n_hidden_units
        self.n_outputs = n_outputs

        self.alpha = alpha

        # TODO: Define a hidden layer and the output layer
        self.l1 = Layer(num_inputs, 1, Sigmoid) # hidden layer 1
        self.l_out = Layer(num_inputs, 1, LinearActivation) # output layer

    def predict(self, x):
        """ 
        Forward pass prediction given the input x
        TODO: Write the function
        """
        # print(f"mlp: {x}")
        a = self.l1.predict(x)
        print(a)
        # print("done with hidden")
        output = self.l_out.predict(a)
        return output

    def export_weights(self):
        return [self.l1.w, self.l_out.w]

    def import_weights(self, ws):
        if ws[0].shape == (self.n_inputs+1, self.n_hidden_units) and ws[1].shape == (self.n_hidden_units+1, 1):
            #print("Importing weights..")
            self.l1.import_weights(ws[0])
            self.l_out.import_weights(ws[1])        
        else:
            print("Sizes do not match")

    def train(self, inputs, outputs):
        """
            Train the network

        Parameters
        ----------
        `x` : numpy array
            Inputs (size: n_examples, n_inputs)
        `t` : numpy array
            Targets (size: n_examples, n_outputs)

        TODO: Write the function to iterate through training examples and apply gradient descent to update the neuron weights
        """

        # delta1 = np.array([[0.0,0.0],[0.0,0.0],[0.0,0.0]])
        # delta_out = np.array([[0.0],[0.0],[0.0]])
        w_hidden = np.random.uniform(size=(self.num_inputs, self.n_hidden_units))
        w_output = np.random.uniform(size=(self.n_hidden_units, self.n_outputs))

        d_hidden = np.random.uniform(size=(self.num_inputs, self.n_hidden_units))
        d_output = np.random.uniform(size=(self.n_hidden_units, self.n_outputs))

        a_array = []
        o_array = []

        r = self.alpha      # Learning rate
        N = len(inputs)

        w1 = self.l1.w
        w2 = self.l_out.w

        # weights = self.import_weights()

        # Loop over training examples
        for input, target in zip(inputs, outputs):
            # Forward pass
            a1 = self.l1.activation(input)
            # print(f"Activations for layer 1: {a1}")
            a_array.append(a1[0])
            o1 = self.l1.output(a1)
            # print(f"Outputs for layer 1: {o1}")
            o_array.append(o1[0])
            # delta1_error = o1-target

            a2 = self.l_out.activation(o1)
            # print(f"Activations for layer 2: {a2}")
            a_array.append(a2[0])
            y = self.l_out.output(a2)
            # print(f"Output of output layer: {y}")
            o_array.append(y[0])

            ## Error calculation
            error1 = target - o1
            error2 = target-y

            ## Backpropogation
            # Layer 1
            dJ1 = self.l1.gradient(a1)*error1
            d_hidden += dJ1
            # w_hidden += o1
            w_hidden += (-r/N)*o1.T.dot(dJ1)

            # Layer 2
            dJ2 = self.l_out.gradient(a2)*error2
            print(f"ERROR: {dJ2}")
            d_output += dJ2
            # w_output += y
            w_output += (-r/N)*y.T.dot(dJ2)

        # print(f"hidden: w {w_hidden}, d {d_hidden}")
        # print(f"output: w {w_output}, d {d_output}")

        # hidden_sum = w_hidden.T.dot(d_hidden)
        # # print(f"hidden_sum: {hidden_sum}")
        # w1 = w1 + np.sum((-r/N)*d_hidden.T.dot(hidden_sum))
        # # output_sum = w_output.T.dot(d_output)
        # # print(f"output_sum: {output_sum}")
        # # w2 = (-r/N)*d_output.T.dot(output_sum)
        # # output_sum = w_output.T.dot(d_output)
        # # print(f"w_output: {w_output}")
        # w2 = w2 + np.sum((-r/N)*d_output.T.dot(w_output))
        # # self.l1.w += w_hidden
        # # self.l_out += w_output
        # print(f"RESULTS: {w1}, {w2}")
            
        

        hidden_sum = w_hidden.T.dot(d_hidden)
        # w1 = w1 + np.sum((-r/N)*d_hidden.T.dot(hidden_sum))
        # w2 = w2 + np.sum((-r/N)*d_output.T.dot(w_output)) + w1
        w1 = w1 + np.sum(d_hidden)
        w2 = w2 + np.sum(d_output) + w1
        # print(f"{w1}, {w2}")
        print(f"RESULTS: {w2}")

        # print(f"Here are all the activation values: {a_array}")
        # print(f"Here are all the output values: {o_array}")
            
    def calc_prediction_error(self, model, x, t):
        """ Calculate the average prediction error """
        E = 0
        for input, target in zip(x, t):     # x = training data, t = training target
            output = model.predict(input)
            print(f"OUTPUT: {output}, TARGET: {target}")
            delta_w = (target - output)
            print(f"ERROR: {delta_w}")
            E = E + delta_w**2
        E = E/len(t)
        return E

In [96]:
data = np.array( [ [0.5, 0.5, 0], [1.0, 0, 0], [2.0, 3.0, 0], [0, 1.0, 1], [0, 2.0, 1], [1.0, 2.2, 1] ] )
xdata = data[:,:2]
ydata = data[:,2]
mlp3 = MLP(num_inputs, n_hidden_units, n_outputs, alpha=1e-3)

num_epochs = 5
for epoch in range(num_epochs):
    print(f"EPOCH: {epoch}")
    mlp3.train(xdata, ydata)

EPOCH: 0
ERROR: [-1.]
ERROR: [-1.]
ERROR: [-1.]
ERROR: [0.]
ERROR: [0.]
ERROR: [0.]
RESULTS: [[-10.07775726]
 [-10.83842321]]
EPOCH: 1
ERROR: [-1.]
ERROR: [-1.]
ERROR: [-1.]
ERROR: [0.]
ERROR: [0.]
ERROR: [0.]
RESULTS: [[-10.3935117 ]
 [-11.15417766]]
EPOCH: 2
ERROR: [-1.]
ERROR: [-1.]
ERROR: [-1.]
ERROR: [0.]
ERROR: [0.]
ERROR: [0.]
RESULTS: [[ -9.26229769]
 [-10.02296364]]
EPOCH: 3
ERROR: [-1.]
ERROR: [-1.]
ERROR: [-1.]
ERROR: [0.]
ERROR: [0.]
ERROR: [0.]
RESULTS: [[ -9.71678396]
 [-10.47744992]]
EPOCH: 4
ERROR: [-1.]
ERROR: [-1.]
ERROR: [-1.]
ERROR: [0.]
ERROR: [0.]
ERROR: [0.]
RESULTS: [[ -9.75697536]
 [-10.51764131]]


The results are not decreasing enough for us.

7. Train your network to act like an XOR gate.

We were unable to implement this since our training function wasn't working properly but the set up is already there to implement this and test out different learning rates and number of epochs. Then we would map it out just like with 1.1